In [1]:
import ee
import geemap
import datetime

In [2]:
ee.Authenticate()

True

In [3]:
ee.Initialize(project='black-octagon-291810')

**Dataset**

* The Dynamic World (DW): a continuously updating Image Collection of globally consistent, 10m-resolution, near real-time (NRT) land use land cover (LULC) predictions created from Sentinel-2 imagery.

* Images in this dataset include 10 bands: 9 bands with estimated probabilities for each of the nine DW LULC classes and a class "label" band indicating the class with the largest estimated probability.


# Area of Interests & geometries for Singapore and Dar Es Salaam

In [4]:
# Dar es Salaam
aoi_dar = ee.FeatureCollection("projects/sat-io/open-datasets/FAO/GAUL/GAUL_2024_L1") \
            .filter(ee.Filter.eq("gaul1_name", "Dar Es Salaam"))
dar_geom = aoi_dar.geometry()

# Singapore
aoi_sgp = ee.FeatureCollection("projects/sat-io/open-datasets/FAO/GAUL/GAUL_2024_L0") \
            .filter(ee.Filter.eq("gaul0_name", "Singapore"))
sgp_geom = aoi_sgp.geometry()

# Generate Sentinel-2 images for SGP and DAR in 2020 & 2025
https://developers.google.com/earth-engine/tutorials/community/introduction-to-dynamic-world-pt-1#load_a_sentinel-2_image

find a Sentinel-2 image by applying filters on the Sentinel-2 L1C harmonized collection for the date range and location of interest. Since the Dynamic World classification is available only for scenes with < 35% cloud cover, we also apply a metadata filter.

In [5]:
def get_sentinel2(geometry, year):
    start_date = f'{year}-01-01'
    end_date = f'{year}-12-31'

    s2 = (
        ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
        .filterDate(start_date, end_date)
        .filterBounds(geometry)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) # Relax cloud filter to 20%
        )

    # call .first() to extract a single image (the earliest one matching our criteria) from the collection

    return s2.median().clip(geometry) # Use median composite and clip

# Generate Sentinel-2 Image for Dar es Salaam (2020 vs 2025)
s2_dar_2020 = get_sentinel2(dar_geom, 2020)
s2_dar_2025 = get_sentinel2(dar_geom, 2025)

# Generate Sentinel-2 for Singapore (2020 vs 2025)
s2_sgp_2020 = get_sentinel2(sgp_geom, 2020)
s2_sgp_2025 = get_sentinel2(sgp_geom, 2025)

Once we have the image, we can add it to the map to visualize it

## Visualize Sentinel-2 images

In [6]:
Map = geemap.Map()

# Sentinel2 Vis
s2VisParams = {
    "bands": ['B4', 'B3', 'B2'],
    "min": 0,
    "max": 3000
}

# Dar es Salaam S2 Layers
Map.addLayer(s2_dar_2020, s2VisParams, 'DAR Sentinel-2 2020')
Map.addLayer(s2_dar_2025, s2VisParams, 'DAR Sentinel-2 2025')

# Singapore S2 Layers
Map.addLayer(s2_sgp_2020, s2VisParams, 'SGP Sentinel-2 2020')
Map.addLayer(s2_sgp_2025, s2VisParams, 'SGP Sentinel-2 2025')

# Set center
Map.centerObject(sgp_geom, 11) # (dar_geom, 10) or (sgp_geom, 11)
Map

Map(center=[1.361135578765903, 103.8202830927031], controls=(WidgetControl(options=['position', 'transparent_b…

# Find the Matching Dynamic World Image

https://developers.google.com/earth-engine/tutorials/community/introduction-to-dynamic-world-pt-1#find_the_matching_dynamic_world_image

To find the matching classified image in the Dynamic World collection, we need to **extract the product id** using the system:index property.

In [7]:
imageId_dar_2020 = s2_dar_2020.get('system:index')
imageId_dar_2025 = s2_dar_2025.get('system:index')

imageId_sgp_2020 = s2_sgp_2020.get('system:index')
imageId_sgp_2025 = s2_sgp_2025.get('system:index')

print(imageId_dar_2020)

ee.ComputedObject({
  "functionInvocationValue": {
    "functionName": "Element.get",
    "arguments": {
      "object": {
        "functionInvocationValue": {
          "functionName": "Image.clip",
          "arguments": {
            "geometry": {
              "functionInvocationValue": {
                "functionName": "Collection.geometry",
                "arguments": {
                  "collection": {
                    "functionInvocationValue": {
                      "functionName": "Collection.filter",
                      "arguments": {
                        "collection": {
                          "functionInvocationValue": {
                            "functionName": "Collection.loadTable",
                            "arguments": {
                              "tableId": {
                                "constantValue": "projects/sat-io/open-datasets/FAO/GAUL/GAUL_2024_L1"
                              }
                            }
                          }

The Sentinel-2 product IDs are printed in the consoles above

We can **use the same id to load the matching Dynamic World scene**. The code snippet below applies a filter on the Dynamic World collection and extracts the matching scene.

In [8]:
# Dar es Salaam DW scene in 2020
dw_dar_2020 = (
    ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
    .filter(ee.Filter.eq('system:index', imageId_dar_2020))
)
dwImage_dar_2020 = ee.Image(dw_dar_2020.first())

# Dar es Salaam DW scene in 2025
dw_dar_2025 = (
    ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
    .filter(ee.Filter.eq('system:index', imageId_dar_2025))
)
dwImage_dar_2025 = ee.Image(dw_dar_2025.first())

# Singapore DW scene in 2020
dw_sgp_2020 = (
    ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
    .filter(ee.Filter.eq('system:index', imageId_sgp_2020))
)
dwImage_sgp_2020 = ee.Image(dw_sgp_2020.first())

# Singapore DW scene in 2025
dw_sgp_2025 = (
    ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
    .filter(ee.Filter.eq('system:index', imageId_sgp_2025))
)
dwImage_sgp_2025 = ee.Image(dw_sgp_2025.first())

print(dwImage_dar_2020)

ee.Image({
  "functionInvocationValue": {
    "functionName": "Collection.first",
    "arguments": {
      "collection": {
        "functionInvocationValue": {
          "functionName": "Collection.filter",
          "arguments": {
            "collection": {
              "functionInvocationValue": {
                "functionName": "ImageCollection.load",
                "arguments": {
                  "id": {
                    "constantValue": "GOOGLE/DYNAMICWORLD/V1"
                  }
                }
              }
            },
            "filter": {
              "functionInvocationValue": {
                "functionName": "Filter.equals",
                "arguments": {
                  "leftField": {
                    "constantValue": "system:index"
                  },
                  "rightValue": {
                    "functionInvocationValue": {
                      "functionName": "Element.get",
                      "arguments": {
                        "ob

# Generate Dynamic World images

https://developers.google.com/earth-engine/tutorials/community/introduction-to-dynamic-world-pt-1#visualize_the_classified_image



The label band of Dynamic World images contains a discrete label for each pixel assigned based on the class with the highest probability.

In [9]:
# Helper function to get a median Dynamic World LULC image for a given year
def get_lulc_image(year, geometry):
    start_date = f'{year}-01-01'
    end_date = f'{year}-12-31'

    collection = (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterDate(start_date, end_date)
        .filterBounds(geometry)
    )

    classification = collection.select('label')

     # get median image and clipped using geometry
    return classification.median().clip(geometry)

## Visualize Dynamic World images (classified)

In [10]:
dwVisParams = {
  'min': 0,
  'max': 8,
  'palette': [
    '#419BDF', '#397D49', '#88B053', '#7A87C6', '#E49635', '#DFC35A',
    '#C4281B', '#A59B8F', '#B39FE1'
  ]
};

# Generate DW for Dar es Salaam with classification
lulc_2020_dar = get_lulc_image(2020, dar_geom)
lulc_2025_dar = get_lulc_image(2025, dar_geom)

# Generate DW for Singapore with classification
lulc_2020_sgp = get_lulc_image(2020, sgp_geom)
lulc_2025_sgp = get_lulc_image(2025, sgp_geom)

# Add layers
Map.addLayer(lulc_2020_dar, dwVisParams, 'DAR 2020 LULC Classified Image')
Map.addLayer(lulc_2025_dar, dwVisParams, 'DAR 2025 LULC Classified Image')
Map.addLayer(lulc_2020_sgp, dwVisParams, 'SGP 2020 LULC Classified Image')
Map.addLayer(lulc_2025_sgp, dwVisParams, 'SGP 2025 LULC Classified Image')

# Set object center
Map.centerObject(sgp_geom, 11) # (dar_geom, 10) or (sgp_geom, 11)
# Map.add_legend(title="Dynamic World V1 Land Cover", builtin_legend="Dynamic_World")
Map

Map(bottom=260462.0, center=[1.361135578765903, 103.8202830927031], controls=(WidgetControl(options=['position…

# Create a Mode Composite

In [11]:
def get_mode_composite(year, geometry):
    start_date = f'{year}-01-01'
    end_date = f'{year}-12-31'

    collection = (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterBounds(geometry)
        .filterDate(start_date, end_date)
    )

    classification = collection.select('label');

     # get median image and clipped using geometry
    return classification.median().clip(geometry).reduce(ee.Reducer.mode()).rename('dw_mode_classification')

In [12]:
# Generate DW Composite
dwComposite_dar_2020 = get_mode_composite(2020, dar_geom)
dwComposite_dar_2025 = get_mode_composite(2025, dar_geom)
dwComposite_sgp_2020 = get_mode_composite(2020, sgp_geom)
dwComposite_sgp_2025 = get_mode_composite(2025, sgp_geom)

# **Statistics for Built Area only**

## Built Area in Dar es Salaam, 2020

#### calculation:

In [13]:
builtArea_dar_2020 = dwComposite_dar_2020.eq(6).rename('built_area_mask')

totalPixels_dar_2020 = dwComposite_dar_2020.reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=dar_geom,
    scale=10,
    maxPixels=1e10
).get('dw_mode_classification')

builtAreaPixels_dar_2020 = builtArea_dar_2020.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=dar_geom,
    scale=10,
    maxPixels=1e10
).get('built_area_mask')

In [14]:
# Pixel area (scale is 10m)
pixel_area_sq_m = 10 * 10
pixel_area_sq_km = pixel_area_sq_m / 1_000_000

# Get the numerical values from Earth Engine objects
built_area_pixels_dar_2020 = ee.Number(builtAreaPixels_dar_2020).getInfo()
total_pixels_dar_2020 = ee.Number(totalPixels_dar_2020).getInfo()

# Calculate built area in square kilometers
built_area_sqkm_dar_2020 = built_area_pixels_dar_2020 * pixel_area_sq_km

# Calculate total area in square kilometers
total_area_sqkm_dar_2020 = total_pixels_dar_2020 * pixel_area_sq_km

# Calculate percentage of built area
pct_built_area_dar_2020 = (built_area_sqkm_dar_2020 / total_area_sqkm_dar_2020) * 100 if total_area_sqkm_dar_2020 > 0 else 0

#### **result:**

In [15]:
print(f"Built Area in DAR (2020): {built_area_sqkm_dar_2020:.2f} sq km")
print(f"Total Area in DAR (2020): {total_area_sqkm_dar_2020:.2f} sq km")
print(f"Percentage Built Area in DAR (2020) compared to all classifications: {pct_built_area_dar_2020:.2f}%")

Built Area in DAR (2020): 938.91 sq km
Total Area in DAR (2020): 1631.40 sq km
Percentage Built Area in DAR (2020) compared to all classifications: 57.55%


## Built Area in Dar es Salaam, 2025

#### calculation:

In [16]:
builtArea_dar_2025 = dwComposite_dar_2025.eq(6).rename('built_area_mask')

totalPixels_dar_2025 = dwComposite_dar_2025.reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=dar_geom,
    scale=10,
    maxPixels=1e10
).get('dw_mode_classification')

builtAreaPixels_dar_2025 = builtArea_dar_2025.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=dar_geom,
    scale=10,
    maxPixels=1e10
).get('built_area_mask')

In [17]:
# Get the numerical values from Earth Engine objects
built_area_pixels_dar_2025 = ee.Number(builtAreaPixels_dar_2025).getInfo()
total_pixels_dar_2025 = ee.Number(totalPixels_dar_2025).getInfo()

# Calculate built area in square kilometers
built_area_sqkm_dar_2025 = built_area_pixels_dar_2025 * pixel_area_sq_km

# Calculate total area in square kilometers
total_area_sqkm_dar_2025 = total_pixels_dar_2025 * pixel_area_sq_km

# Calculate percentage of built area
pct_built_area_dar_2025 = (built_area_sqkm_dar_2025 / total_area_sqkm_dar_2025) * 100 if total_area_sqkm_dar_2025 > 0 else 0

#### **result:**

In [18]:
print(f"Built Area in DAR (2025): {built_area_sqkm_dar_2025:.2f} sq km")
print(f"Total Area in DAR (2025): {total_area_sqkm_dar_2025:.2f} sq km")
print(f"Percentage Built Area in DAR (2025) compared to all classifications: {pct_built_area_dar_2025:.2f}%")

Built Area in DAR (2025): 1001.09 sq km
Total Area in DAR (2025): 1638.36 sq km
Percentage Built Area in DAR (2025) compared to all classifications: 61.10%


## Built Area in Singapore, 2020



#### calculation:

In [19]:
builtArea_sgp_2020 = dwComposite_sgp_2020.eq(6).rename('built_area_mask')

totalPixels_sgp_2020 = dwComposite_sgp_2020.reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=sgp_geom,
    scale=10,
    maxPixels=1e10
).get('dw_mode_classification')

builtAreaPixels_sgp_2020 = builtArea_sgp_2020.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=sgp_geom,
    scale=10,
    maxPixels=1e10
).get('built_area_mask')

In [20]:
# Pixel area (scale is 10m)
pixel_area_sq_m = 10 * 10
pixel_area_sq_km = pixel_area_sq_m / 1_000_000

# Get the numerical values from Earth Engine objects
built_area_pixels_sgp_2020 = ee.Number(builtAreaPixels_sgp_2020).getInfo()
total_pixels_sgp_2020 = ee.Number(totalPixels_sgp_2020).getInfo()

# Calculate built area in square kilometers
built_area_sqkm_sgp_2020 = built_area_pixels_sgp_2020 * pixel_area_sq_km

# Calculate total area in square kilometers
total_area_sqkm_sgp_2020 = total_pixels_sgp_2020 * pixel_area_sq_km

# Calculate percentage of built area
pct_built_area_sgp_2020 = (built_area_sqkm_sgp_2020 / total_area_sqkm_sgp_2020) * 100 if total_area_sqkm_sgp_2020 > 0 else 0

#### **result:**

In [21]:
print(f"Built Area in SGP (2020): {built_area_sqkm_sgp_2020:.2f} sq km")
print(f"Total Area in SGP (2020): {total_area_sqkm_sgp_2020:.2f} sq km")
print(f"Percentage Built Area in SGP (2020) compared to all classifications: {pct_built_area_sgp_2020:.2f}%")

Built Area in SGP (2020): 324.15 sq km
Total Area in SGP (2020): 586.10 sq km
Percentage Built Area in SGP (2020) compared to all classifications: 55.31%


## Built Area in Singapore, 2025


#### calculation:



In [22]:
builtArea_sgp_2025 = dwComposite_sgp_2025.eq(6).rename('built_area_mask')

totalPixels_sgp_2025 = dwComposite_sgp_2025.reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=sgp_geom,
    scale=10,
    maxPixels=1e10
).get('dw_mode_classification')

builtAreaPixels_sgp_2025 = builtArea_sgp_2025.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=sgp_geom,
    scale=10,
    maxPixels=1e10
).get('built_area_mask')

In [23]:
# Pixel area (scale is 10m)
pixel_area_sq_m = 10 * 10
pixel_area_sq_km = pixel_area_sq_m / 1_000_000

# Get the numerical values from Earth Engine objects
built_area_pixels_sgp_2025 = ee.Number(builtAreaPixels_sgp_2025).getInfo()
total_pixels_sgp_2025 = ee.Number(totalPixels_sgp_2025).getInfo()

# Calculate built area in square kilometers
built_area_sqkm_sgp_2025 = built_area_pixels_sgp_2025 * pixel_area_sq_km

# Calculate total area in square kilometers
total_area_sqkm_sgp_2025 = total_pixels_sgp_2025 * pixel_area_sq_km

# Calculate percentage of built area
pct_built_area_sgp_2025 = (built_area_sqkm_sgp_2025 / total_area_sqkm_sgp_2025) * 100 if total_area_sqkm_sgp_2025 > 0 else 0

#### **result:**

In [24]:
print(f"Built Area in SGP (2025): {built_area_sqkm_sgp_2025:.2f} sq km")
print(f"Total Area in SGP (2025): {total_area_sqkm_sgp_2025:.2f} sq km")
print(f"Percentage Built Area in SGP (2025) compared to all classifications: {pct_built_area_sgp_2025:.2f}%")

Built Area in SGP (2025): 319.30 sq km
Total Area in SGP (2025): 596.10 sq km
Percentage Built Area in SGP (2025) compared to all classifications: 53.56%


# **Statistics for All LULC Classes**

In [25]:
classLabels = [
    'water', 'trees', 'grass', 'flooded_vegetation', 'crops',
    'shrub_and_scrub', 'built', 'bare', 'snow_and_ice'
];

## LULC statistics in **Dar es Salaam, 2020**

#### calculation:

In [26]:
pixelCountStats_dar_2020 = dwComposite_dar_2020.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram().unweighted(),
    geometry=dar_geom,
    scale=10,
    maxPixels=1e10
)

pixelCounts_dar_2020 = ee.Dictionary(pixelCountStats_dar_2020.get('dw_mode_classification'));
print(pixelCounts_dar_2020)

ee.Dictionary({
  "functionInvocationValue": {
    "functionName": "Dictionary",
    "arguments": {
      "input": {
        "functionInvocationValue": {
          "functionName": "Dictionary.get",
          "arguments": {
            "dictionary": {
              "functionInvocationValue": {
                "functionName": "Image.reduceRegion",
                "arguments": {
                  "geometry": {
                    "functionInvocationValue": {
                      "functionName": "Collection.geometry",
                      "arguments": {
                        "collection": {
                          "functionInvocationValue": {
                            "functionName": "Collection.filter",
                            "arguments": {
                              "collection": {
                                "functionInvocationValue": {
                                  "functionName": "Collection.loadTable",
                                  "arguments": {
         

In [27]:
# Get the actual numerical values from the Earth Engine dictionary
raw_counts_py_dar_2020 = pixelCounts_dar_2020.getInfo()

# Initialize a new Python dictionary with formatted labels and zero counts
numerical_pixel_counts_dar_2020 = {label: 0 for label in classLabels}

# Aggregate counts for each class, handling float keys from Earth Engine
for key, count in raw_counts_py_dar_2020.items():
    if key == 'null': # Handle null values if present, typically from masked areas
        continue
    try:
        # Convert the float string key to an integer class index
        class_index = int(float(key))
        if 0 <= class_index < len(classLabels):
            numerical_pixel_counts_dar_2020[classLabels[class_index]] += count
    except ValueError:
        # Skip keys that cannot be converted to float (e.g., unexpected strings)
        continue

print(numerical_pixel_counts_dar_2020);

{'water': 144469, 'trees': 2719670, 'grass': 450223, 'flooded_vegetation': 650794, 'crops': 310663, 'shrub_and_scrub': 2547478, 'built': 9399594, 'bare': 90145, 'snow_and_ice': 1004}


#### **result:**

In [28]:
print("\nArea and Percentage of each class in square kilometers (Dar es Salaam, 2020):")
total_area_dar_2020_sq_km = ee.Number(totalPixels_dar_2020).getInfo() * pixel_area_sq_km # Get the numerical value for total area

for class_name, pixel_count in numerical_pixel_counts_dar_2020.items():
    area_sq_km = pixel_count * pixel_area_sq_km
    percentage = (area_sq_km / total_area_dar_2020_sq_km) * 100 if total_area_dar_2020_sq_km > 0 else 0
    print(f"{class_name}: {area_sq_km:.2f} sq km ({percentage:.2f}%)")


Area and Percentage of each class in square kilometers (Dar es Salaam, 2020):
water: 14.45 sq km (0.89%)
trees: 271.97 sq km (16.67%)
grass: 45.02 sq km (2.76%)
flooded_vegetation: 65.08 sq km (3.99%)
crops: 31.07 sq km (1.90%)
shrub_and_scrub: 254.75 sq km (15.62%)
built: 939.96 sq km (57.62%)
bare: 9.01 sq km (0.55%)
snow_and_ice: 0.10 sq km (0.01%)


## LULC statistics in **Dar es Salaam, 2025**

#### calculation:

In [29]:
pixelCountStats_dar_2025 = dwComposite_dar_2025.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram().unweighted(),
    geometry=dar_geom,
    scale=10,
    maxPixels=1e10
)

pixelCounts_dar_2025 = ee.Dictionary(pixelCountStats_dar_2025.get('dw_mode_classification'))
print(pixelCounts_dar_2025)

ee.Dictionary({
  "functionInvocationValue": {
    "functionName": "Dictionary",
    "arguments": {
      "input": {
        "functionInvocationValue": {
          "functionName": "Dictionary.get",
          "arguments": {
            "dictionary": {
              "functionInvocationValue": {
                "functionName": "Image.reduceRegion",
                "arguments": {
                  "geometry": {
                    "functionInvocationValue": {
                      "functionName": "Collection.geometry",
                      "arguments": {
                        "collection": {
                          "functionInvocationValue": {
                            "functionName": "Collection.filter",
                            "arguments": {
                              "collection": {
                                "functionInvocationValue": {
                                  "functionName": "Collection.loadTable",
                                  "arguments": {
         

In [30]:
# Get the actual numerical values from the Earth Engine dictionary
raw_counts_py_dar_2025 = pixelCounts_dar_2025.getInfo()

# Initialize a new Python dictionary with formatted labels and zero counts
numerical_pixel_counts_dar_2025 = {label: 0 for label in classLabels}

# Aggregate counts for each class, handling float keys from Earth Engine
for key, count in raw_counts_py_dar_2025.items():
    if key == 'null': # Handle null values if present, typically from masked areas
        continue
    try:
        # Convert the float string key to an integer class index
        class_index = int(float(key))
        if 0 <= class_index < len(classLabels):
            numerical_pixel_counts_dar_2025[classLabels[class_index]] += count
    except ValueError:
        # Skip keys that cannot be converted to float (e.g., unexpected strings)
        continue

print(numerical_pixel_counts_dar_2025)

{'water': 133638, 'trees': 2326509, 'grass': 350034, 'flooded_vegetation': 477807, 'crops': 212373, 'shrub_and_scrub': 2744627, 'built': 10018178, 'bare': 120001, 'snow_and_ice': 401}


### **result:**

In [31]:
print("\nArea and Percentage of each class in square kilometers (Dar es Salaam, 2025):")
total_area_dar_2025_sq_km = ee.Number(totalPixels_dar_2025).getInfo() * pixel_area_sq_km # Get the numerical value for total area

for class_name, pixel_count in numerical_pixel_counts_dar_2025.items():
    area_sq_km = pixel_count * pixel_area_sq_km
    percentage = (area_sq_km / total_area_dar_2025_sq_km) * 100 if total_area_dar_2025_sq_km > 0 else 0
    print(f"{class_name}: {area_sq_km:.2f} sq km ({percentage:.2f}%)")


Area and Percentage of each class in square kilometers (Dar es Salaam, 2025):
water: 13.36 sq km (0.82%)
trees: 232.65 sq km (14.20%)
grass: 35.00 sq km (2.14%)
flooded_vegetation: 47.78 sq km (2.92%)
crops: 21.24 sq km (1.30%)
shrub_and_scrub: 274.46 sq km (16.75%)
built: 1001.82 sq km (61.15%)
bare: 12.00 sq km (0.73%)
snow_and_ice: 0.04 sq km (0.00%)


## LULC statistics in **Singapore, 2020**

#### calculation:

In [32]:
pixelCountStats_sgp_2020 = dwComposite_sgp_2020.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram().unweighted(),
    geometry=sgp_geom,
    scale=10,
    maxPixels=1e10
)

pixelCounts_sgp_2020 = ee.Dictionary(pixelCountStats_sgp_2020.get('dw_mode_classification'));
print(pixelCounts_sgp_2020)

ee.Dictionary({
  "functionInvocationValue": {
    "functionName": "Dictionary",
    "arguments": {
      "input": {
        "functionInvocationValue": {
          "functionName": "Dictionary.get",
          "arguments": {
            "dictionary": {
              "functionInvocationValue": {
                "functionName": "Image.reduceRegion",
                "arguments": {
                  "geometry": {
                    "functionInvocationValue": {
                      "functionName": "Collection.geometry",
                      "arguments": {
                        "collection": {
                          "functionInvocationValue": {
                            "functionName": "Collection.filter",
                            "arguments": {
                              "collection": {
                                "functionInvocationValue": {
                                  "functionName": "Collection.loadTable",
                                  "arguments": {
         

In [33]:
# Get the actual numerical values from the Earth Engine dictionary
raw_counts_py_sgp_2020 = pixelCounts_sgp_2020.getInfo()

# Initialize a new Python dictionary with formatted labels and zero counts
numerical_pixel_counts_sgp_2020 = {label: 0 for label in classLabels}

# Aggregate counts for each class, handling float keys from Earth Engine
for key, count in raw_counts_py_sgp_2020.items():
    if key == 'null': # Handle null values if present, typically from masked areas
        continue
    try:
        # Convert the float string key to an integer class index
        class_index = int(float(key))
        if 0 <= class_index < len(classLabels):
            numerical_pixel_counts_sgp_2020[classLabels[class_index]] += count
    except ValueError:
        # Skip keys that cannot be converted to float (e.g., unexpected strings)
        continue

print(numerical_pixel_counts_sgp_2020);

{'water': 515114, 'trees': 1557023, 'grass': 252651, 'flooded_vegetation': 76951, 'crops': 56122, 'shrub_and_scrub': 33278, 'built': 3267295, 'bare': 101652, 'snow_and_ice': 919}


#### **result:**

In [34]:
print("\nArea and Percentage of each class in square kilometers (Singapore, 2020):")
total_area_sgp_2020_sq_km = ee.Number(totalPixels_sgp_2020).getInfo() * pixel_area_sq_km # Get the numerical value for total area

for class_name, pixel_count in numerical_pixel_counts_sgp_2020.items():
    area_sq_km = pixel_count * pixel_area_sq_km
    percentage = (area_sq_km / total_area_sgp_2020_sq_km) * 100 if total_area_sgp_2020_sq_km > 0 else 0
    print(f"{class_name}: {area_sq_km:.2f} sq km ({percentage:.2f}%)")


Area and Percentage of each class in square kilometers (Singapore, 2020):
water: 51.51 sq km (8.79%)
trees: 155.70 sq km (26.57%)
grass: 25.27 sq km (4.31%)
flooded_vegetation: 7.70 sq km (1.31%)
crops: 5.61 sq km (0.96%)
shrub_and_scrub: 3.33 sq km (0.57%)
built: 326.73 sq km (55.75%)
bare: 10.17 sq km (1.73%)
snow_and_ice: 0.09 sq km (0.02%)


## LULC statistics in **Singapore, 2025**

#### calculation

In [35]:
pixelCountStats_sgp_2025 = dwComposite_sgp_2025.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram().unweighted(),
    geometry=sgp_geom,
    scale=10,
    maxPixels=1e10
)

pixelCounts_sgp_2025 = ee.Dictionary(pixelCountStats_sgp_2025.get('dw_mode_classification'));
print(pixelCounts_sgp_2025)

ee.Dictionary({
  "functionInvocationValue": {
    "functionName": "Dictionary",
    "arguments": {
      "input": {
        "functionInvocationValue": {
          "functionName": "Dictionary.get",
          "arguments": {
            "dictionary": {
              "functionInvocationValue": {
                "functionName": "Image.reduceRegion",
                "arguments": {
                  "geometry": {
                    "functionInvocationValue": {
                      "functionName": "Collection.geometry",
                      "arguments": {
                        "collection": {
                          "functionInvocationValue": {
                            "functionName": "Collection.filter",
                            "arguments": {
                              "collection": {
                                "functionInvocationValue": {
                                  "functionName": "Collection.loadTable",
                                  "arguments": {
         

In [36]:
# Get the actual numerical values from the Earth Engine dictionary
raw_counts_py_sgp_2025 = pixelCounts_sgp_2025.getInfo()

# Initialize a new Python dictionary with formatted labels and zero counts
numerical_pixel_counts_sgp_2025 = {label: 0 for label in classLabels}

# Aggregate counts for each class, handling float keys from Earth Engine
for key, count in raw_counts_py_sgp_2025.items():
    if key == 'null': # Handle null values if present, typically from masked areas
        continue
    try:
        # Convert the float string key to an integer class index
        class_index = int(float(key))
        if 0 <= class_index < len(classLabels):
            numerical_pixel_counts_sgp_2025[classLabels[class_index]] += count
    except ValueError:
        # Skip keys that cannot be converted to float (e.g., unexpected strings)
        continue

print(numerical_pixel_counts_sgp_2025);

{'water': 524949, 'trees': 1737360, 'grass': 196504, 'flooded_vegetation': 120689, 'crops': 36900, 'shrub_and_scrub': 41043, 'built': 3207455, 'bare': 91674, 'snow_and_ice': 4415}


#### **result:**

In [37]:
print("\nArea and Percentage of each class in square kilometers (Singapore, 2025):")
total_area_sgp_2025_sq_km = ee.Number(totalPixels_sgp_2025).getInfo() * pixel_area_sq_km # Get the numerical value for total area

for class_name, pixel_count in numerical_pixel_counts_sgp_2025.items():
    area_sq_km = pixel_count * pixel_area_sq_km
    percentage = (area_sq_km / total_area_sgp_2025_sq_km) * 100 if total_area_sgp_2025_sq_km > 0 else 0
    print(f"{class_name}: {area_sq_km:.2f} sq km ({percentage:.2f}%)")


Area and Percentage of each class in square kilometers (Singapore, 2025):
water: 52.49 sq km (8.81%)
trees: 173.74 sq km (29.15%)
grass: 19.65 sq km (3.30%)
flooded_vegetation: 12.07 sq km (2.02%)
crops: 3.69 sq km (0.62%)
shrub_and_scrub: 4.10 sq km (0.69%)
built: 320.75 sq km (53.81%)
bare: 9.17 sq km (1.54%)
snow_and_ice: 0.44 sq km (0.07%)
